In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_excel('/root/MindPark/backend/app/mobile/service/models/온열질환환자데이터(전처리전).xlsx')
df.head()

,Sex_1male_2female,Age,Weather_1sunny_2cloudy_3rainy_4missing,Location_1outdoor_indoor,Unnamed: 4,HT,HeartDisease,Pscyco,DM,CerevD,...,Plt,BUN,Cre,AST,ALT,CK,CRP,Admission1,ICU,Deadtodischarge
0,1.0,24.0,3.0,NaN,1.0,0,0,0,0,0,...,23.2,17.0,1.69,29.0,48.0,506.0,17.4,1.0,NaN,NaN
1,NaN,43.0,NaN,1.0,NaN,0,0,0,0,0,...,24.3,23.1,3.08,36.0,35.0,883.0,0.46,1.0,NaN,NaN
2,1.0,58.0,NaN,1.0,1.0,0,0,0,0,0,...,31.2,19.4,1.42,33.0,16.0,815.0,0.16,1.0,NaN,NaN
3,1.0,46.0,1.0,1.0,1.0,0,0,0,0,0,...,33.3,32.9,4.20,33.0,54.0,173.0,1,1.0,NaN,NaN
4,1.0,57.0,1.0,1.0,1.0,0,0,0,0,0,...,14.1,21.0,1.33,77.0,36.0,386.0,0.103,1.0,1.0,NaN


In [3]:
df = df.replace({'↓': '', '↑': '', '%': ''}, regex=True)
df = df.apply(pd.to_numeric, errors='coerce')

In [4]:
df= df.drop(columns=['Weather_1sunny_2cloudy_3rainy_4missing','ParkinD', 'Dementia','ICU', 'Deadtodischarge', 'PreGCSlessthan15', 'Abdminal', 'Muscular', 'GCS', 'SBP', 'WBC', 'HR', 'Ht', 'Plt', 'BUN', 'Cre', 'AST', 'ALT', 'CK', 'CRP', 'Unnamed: 4', 'BT', 'PreSBP'])
df.head()

,Sex_1male_2female,Age,Location_1outdoor_indoor,HT,HeartDisease,Pscyco,DM,CerevD,CKD,PreRR,PreBT,PreHR,Admission1
0,1.0,24.0,NaN,0,0,0,0,0,0,18.0,38.0,124.0,1.0
1,NaN,43.0,1.0,0,0,0,0,0,0,22.0,37.3,89.0,1.0
2,1.0,58.0,1.0,0,0,0,0,0,0,18.0,36.5,118.0,1.0
3,1.0,46.0,1.0,0,0,0,0,0,0,18.0,36.2,94.0,1.0
4,1.0,57.0,1.0,0,0,0,0,0,0,30.0,NaN,120.0,1.0


In [5]:
# # 나이 범주화
# df['age'] = (df['Age'] // 10) * 10

# 체온 범주화
bt_bins = [0, 36.0, 37.5, float('inf')]  # (0, 36.0], (36.0, 37.2], (37.2, inf)
bt_labels = [0, 1, 2]

df['PreBT'] = pd.cut(df['PreBT'], bins=bt_bins, labels=bt_labels)
df['PreBT']

# 호흡 범주화
bins = [0, 11, 20, float('inf')]  # <=11, 12~20, >=21
labels = [0, 1, 2]  # 저호흡, 정상, 과호흡

df['PreRR'] = pd.cut(df['PreRR'], bins=bins, labels=labels)

# 심박수 범주화
hr_bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
hr_labels = [0, 1, 2, 3, 4]

# HR을 (220 - 나이)의 비율로 직접 계산해 범주화
df['PreHR'] = pd.cut(df['PreHR'] / (220 - df['Age']), bins=hr_bins, labels=hr_labels, right=False)

In [6]:
from tensorflow import keras
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler

# 결측치가 있는 열만 선택
columns_with_nan = df.columns[df.isnull().any()].tolist()
df_missing = df[columns_with_nan]

# 데이터를 표준화 (오토인코더는 정규화된 입력을 더 잘 학습함)
scaler = StandardScaler()
df_scaled = scaler.fit_transform(df_missing)

# 결측치를 0으로 채운 임시 데이터 (학습을 위해)
df_filled = np.nan_to_num(df_scaled, nan=0.0)

# 오토인코더 모델 생성
input_dim = df_filled.shape[1]
encoding_dim = input_dim // 2  # 절반 크기로 압축

input_layer = Input(shape=(input_dim,))
encoded = Dense(encoding_dim, activation="relu")(input_layer)
decoded = Dense(input_dim, activation="linear")(encoded)

autoencoder = Model(input_layer, decoded)
autoencoder.compile(optimizer="adam", loss="mse")

# 모델 학습
autoencoder.fit(df_filled, df_filled, epochs=100, batch_size=16, verbose=1)

# 결측치 복원
df_reconstructed = autoencoder.predict(df_filled)

# 원래 스케일로 변환
df_imputed = scaler.inverse_transform(df_reconstructed)

# 복원된 데이터에서 결측값을 채움
df[columns_with_nan] = np.where(df_missing.isnull(), df_imputed, df_missing)

2025-06-06 07:32:26.323067: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-06 07:32:26.326566: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-06 07:32:26.335110: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749195146.350865  259011 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749195146.355620  259011 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1749195146.368367  259011 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

Epoch 1/100


2025-06-06 07:32:28.606834: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


199/199 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.8722
Epoch 2/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6736
Epoch 3/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.6004
Epoch 4/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4933
Epoch 5/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.4272
Epoch 6/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3647
Epoch 7/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3488
Epoch 8/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3093
Epoch 9/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3089
Epoch 10/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.3024
Epoch 11/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2835
Epoch 12/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2721
Epoch 13/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2803
Epoch 14/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2799
Epoch 15/100
199/199 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step -

In [7]:
df = df.rename(columns={
    'PreHR': 'hr',
    'PreBT': 'temp',
    'PreRR': 'resp',
    'Sex_1male_2female': 'gender',
    'Age': 'age',
    'Location_1outdoor_indoor': 'location'

})

In [8]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ✅ 데이터 불러오기 및 준비 (Admission1이 종속 변수라고 가정)
x = df.drop(columns=['Admission1'])
y = df['Admission1'].astype(int).to_numpy()

# ✅ 데이터 표준화
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# ✅ 학습/테스트 분리
x_train, x_test, y_train, y_test = train_test_split(x_scaled, y, test_size=0.2, random_state=42)

# ✅ 최적 하이퍼파라미터
best_hidden_layers = 4
best_learning_rate = 0.005
best_batch_size = 32

# ✅ 모델 정의
best_model = Sequential()
best_model.add(Dense(64, input_dim=x.shape[1], activation='relu'))
for _ in range(best_hidden_layers - 1):
    best_model.add(Dense(64, activation='relu'))
best_model.add(Dense(1, activation='sigmoid'))

# ✅ 컴파일
best_model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=best_learning_rate),
    metrics=['accuracy']
)

# ✅ 학습
best_model.fit(x_train, y_train, epochs=30, batch_size=best_batch_size, verbose=1)

# ✅ 예측 (확률 → 이진 클래스 변환)
y_pred_prob = best_model.predict(x_test)
y_pred = (y_pred_prob > 0.5).astype(int)

# 📊 평가 지표 계산
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

# 🖨️ 데이터프레임 형식으로 출력
results_df = pd.DataFrame({
    "지표": ["정확도 (Accuracy)", "정밀도 (Precision)", "재현율 (Recall)", "F1-score"],
    "값 (%)": [f"{acc * 100:.2f}", f"{prec * 100:.2f}", f"{rec * 100:.2f}", f"{f1 * 100:.2f}"]
})

print("\n🎯 최고 성능 모델 평가 결과 (Test Set)")
display(results_df)


/usr/local/lib/python3.9/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7599 - loss: 0.5341
Epoch 2/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8539 - loss: 0.3552
Epoch 3/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8702 - loss: 0.3224
Epoch 4/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8731 - loss: 0.3012
Epoch 5/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8885 - loss: 0.2658
Epoch 6/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8856 - loss: 0.2603
Epoch 7/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8851 - loss: 0.2539
Epoch 8/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8813 - loss: 0.2675
Epoch 9/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8830 - loss: 0.2617
Epoch 10/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8977 - loss: 0.2358
Epoch 11/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8913 - loss: 0.2382
Epoch 12/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9041 - lo

,지표,값 (%)
0,정확도 (Accuracy),88.98
1,정밀도 (Precision),85.36
2,재현율 (Recall),85.36
3,F1-score,85.36


In [9]:
# 모델 저장
best_model.save("heat_illness_model.h5")